In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Define the VAE model
class VAE(nn.Module):
    def __init__(self, image_size=64*64, h_dim=1024, z_dim=20):
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(image_size, h_dim)
        self.fc2 = nn.Linear(h_dim, z_dim)
        self.fc3 = nn.Linear(h_dim, z_dim)
        self.fc4 = nn.Linear(z_dim, h_dim)
        self.fc5 = nn.Linear(h_dim, image_size)
        
    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h), self.fc3(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std
    
    def decode(self, z):
        h = torch.relu(self.fc4(z))
        return torch.sigmoid(self.fc5(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 64*64))
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# Custom dataset to load images from the specified folder
class ImageFolderDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.image_paths = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith(('png', 'jpg', 'jpeg'))]
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('L')  # Convert to grayscale
        if self.transform:
            image = self.transform(image)
        return image
# Parameters
def train_vae(batch_size, learning_rate=1e-4, num_epochs=2000):
    # Transform and DataLoader for real images
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.Grayscale(),  # Convert to grayscale
        transforms.ToTensor()
    ])

    real_images_dir = "./data/"  # Real images folder
    dataset = datasets.ImageFolder(root=real_images_dir, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Model, optimizer, and loss function
    image_size = 64*64
    model = VAE(image_size=image_size)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Loss function
    def loss_function(recon_x, x, mu, logvar):
        BCE = nn.functional.binary_cross_entropy(recon_x, x.view(-1, 64*64), reduction='sum')
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return BCE + KLD

    # Training the VAE on real images
    model.train()
    for epoch in range(num_epochs):
        train_loss = 0
        for batch_idx, (images, _) in enumerate(dataloader):  # (images, labels)
            images = images.view(-1, 64*64)  # Flatten images
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(images)
            loss = loss_function(recon_batch, images, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()

            if batch_idx % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx}/{len(dataloader)}], Loss: {loss.item():.4f}')
        
        print(f'====> Epoch: {epoch+1} Average loss: {train_loss / len(dataloader.dataset):.4f}')

    # Compute and print reconstruction error on the real images
    model.eval()
    reconstruction_error = 0
    with torch.no_grad():
        for images, _ in dataloader:  # Unpack tuple (images, labels)
            images = images.view(-1, 64*64)
            recon_batch, mu, logvar = model(images)
            reconstruction_error += nn.functional.binary_cross_entropy(recon_batch, images, reduction='sum').item()

    reconstruction_error /= len(dataloader.dataset)
    print(f'====> Reconstruction error on real images: {reconstruction_error:.4f}')
    return model, dataloader

# Visualizing and saving some reconstructions
import os

def show_reconstruction(model, dataloader, save_dir="./reconstructed_real/"):
    model.eval()
    os.makedirs(save_dir, exist_ok=True)  # Create folder for saving real images if it doesn't exist
    with torch.no_grad():
        images, _ = next(iter(dataloader))  # Grab a batch of real images
        recon_batch, _, _ = model(images.view(-1, 64*64))
        recon_images = recon_batch.view(-1, 1, 64, 64)  # Reshape to image format

        # Save each reconstructed real image to the specified folder
        for idx in range(recon_images.size(0)):
            recon_image_pil = transforms.ToPILImage()(recon_images[idx].cpu())
            recon_image_pil.save(os.path.join(save_dir, f"reconstructed_real_{idx}.png"))

        # Plot original and reconstructed real images
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title('Original Images (Real)')
        plt.imshow(vutils.make_grid(images[:16], normalize=True, nrow=4).permute(1, 2, 0).numpy())
        plt.savefig("real_images_matrix.png", dpi=300)

        plt.subplot(1, 2, 2)
        plt.title('Reconstructed Images (Real)')
        plt.imshow(vutils.make_grid(recon_images[:16], normalize=True, nrow=4).permute(1, 2, 0).numpy())
        plt.show()

    print(f'Reconstructed real images saved to {save_dir}')


batch_size = 75

# Step 1: Train VAE on real images and visualize reconstructed real images
model, real_dataloader = train_vae(batch_size=batch_size)
show_reconstruction(model, real_dataloader)


In [ ]:

# Part 2: Test VAE with Synthetic Images
def evaluate_synthetic_images(model, batch_size, save_dir="./reconstructed_synthetic/"):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    
    synthetic_image_dir = "./synthetic_images/"  # Synthetic images folder
    dataset = ImageFolderDataset(folder_path=synthetic_image_dir, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    os.makedirs(save_dir, exist_ok=True)  # Create folder for saving synthetic images if it doesn't exist

    synthetic_reconstruction_error = 0  # Initialize error accumulator

    with torch.no_grad():
        for i, images in enumerate(dataloader):  # Unpack images (no labels)
            batch_size_actual = images.size(0)  # Dynamically handle the batch size
            recon_batch, _, _ = model(images.view(batch_size_actual, -1))
            original_images = images.view(batch_size_actual, 1, 64, 64).cpu()
            recon_images = recon_batch.view(batch_size_actual, 1, 64, 64).cpu()

            # Save each reconstructed synthetic image to the specified folder
            for idx in range(recon_images.size(0)):
                recon_image_pil = transforms.ToPILImage()(recon_images[idx].cpu())
                recon_image_pil.save(os.path.join(save_dir, f"reconstructed_synthetic_{i}_{idx}.png"))

            # Plot original and reconstructed synthetic images
            fig, ax = plt.subplots(1, 2)
            ax[0].imshow(vutils.make_grid(original_images[:16], normalize=True, nrow=4).permute(1, 2, 0))
            ax[0].set_title('Original Synthetic Images')
            ax[1].imshow(vutils.make_grid(recon_images[:16], normalize=True, nrow=4).permute(1, 2, 0))
            ax[1].set_title('Reconstructed Synthetic Images')
            plt.show()

            # Compute reconstruction error for this batch of synthetic images
            synthetic_reconstruction_error += nn.functional.binary_cross_entropy(recon_batch, images.view(batch_size_actual, -1), reduction='sum').item()

    # Compute average reconstruction error across all synthetic images
    synthetic_reconstruction_error /= len(dataloader.dataset)
    print(f'====> Reconstruction error on synthetic images: {synthetic_reconstruction_error:.4f}')

    print(f'Reconstructed synthetic images saved to {save_dir}')


# Step 2: Evaluate the synthetic images and visualize reconstruction
evaluate_synthetic_images(model, batch_size=batch_size)
